# Satellite Challenge B: Pick the Best Site for a Solar Farm ☀️

**Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and Technology, University of the West Indies.**

**The question.** Jamaica pays some of the highest electricity prices in the
region, and the island sits in year-round sun. An investor asks your team:
**of three candidate sites, which should get a solar farm, and roughly how many
homes could it power?**

NASA has measured the sunlight falling on every point on Earth, every day,
since 1981, and gives the record away free. Your team is going to use it the
way a real energy analyst would.

### How this hour works

Your team has 45 minutes to build the answer and 60 seconds to present it.

- Cells marked **🚚 JUST RUN** do the heavy coding. Run them, read what they print.
- Cells marked **✏️ YOUR CALL** hold the numbers only your team can decide.
  Every number needs a reason your team can say out loud, and the best reasons
  name a source you found: a website, a news story, a person you asked.
- The last cell prints your team's pitch. Read it to the room.

Judges reward four things: the measurement ran, the chosen numbers have named
sources, the answer is given as a range rather than fake precision, and the
pitch tells a clear story.

---

## Step 1. Measure the sunshine at three sites 🚚 JUST RUN

The cell pulls five years of NASA POWER (Prediction Of Worldwide Energy
Resources) sunlight records for each site, in kWh per square metre per day. A
kWh, a kilowatt-hour, is the unit on the light bill: one kWh runs a fan for
about a day, or an air conditioner for about an hour.

One honest limit to know: NASA POWER's grid squares are about 55 km wide, so
two sites in the same square return the same number. These three sit far
apart on purpose. If your team swaps in its own sites, keep them apart too.

In [ ]:
# Run this once. On Google Colab it takes about a minute.
# If a package is already there, pip will say so and move on.
!pip install -q rasterio requests imageio pandas scikit-learn matplotlib pillow

print("Packages ready.")

In [ ]:
# 🚚 JUST RUN THIS CELL. Nothing to change. It is the toolbox for the whole course.
# ============================================================================
#  JAMAICA EARTH OBSERVATION TOOLKIT
#  Run this cell in every session. It sets up the connection to the satellite
#  archive and defines the handful of functions the whole course uses.
# ============================================================================
import os, math, json, time, warnings
warnings.filterwarnings("ignore")

# GDAL reads the satellite files straight off Amazon's servers over the
# internet. These settings tell it how to behave: no login needed, do not list
# whole directories, retry if the network hiccups.
os.environ.update({
    "AWS_NO_SIGN_REQUEST": "YES",
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "5",
    "GDAL_HTTP_RETRY_DELAY": "2",
})

import requests, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from rasterio.warp import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.vrt import WarpedVRT
from PIL import Image, ImageDraw

STAC_URL = "https://earth-search.aws.element84.com/v1/search"

def _stac_post(url, body, timeout=60, tries=4):
    """POST to the archive, retrying politely if the server is having a moment."""
    for attempt in range(tries):
        try:
            r = requests.post(url, json=body, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if attempt == tries - 1:
                raise
            time.sleep(2 * (attempt + 1))    # 2 s, 4 s, 6 s between tries


# House style for every chart in this course.
CYAN, INK, SAND = "#00b8d4", "#12232e", "#e0a458"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 13,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})

# Places in Jamaica used through the course, as [west, south, east, north].
PLACES = {
    "kingston":     [-76.86, 17.93, -76.80, 18.02],
    "black_river":  [-77.90, 17.96, -77.78, 18.08],
    "negril":       [-78.375, 18.25, -78.320, 18.36],
    "montego_bay":  [-77.97, 18.44, -77.88, 18.51],
    "new_hope":     [-78.20, 18.13, -78.08, 18.24],
    "st_elizabeth": [-77.75, 18.00, -77.65, 18.10],
    "portland":     [-76.45, 18.10, -76.32, 18.20],
    "jamaica":      [-78.45, 17.66, -76.15, 18.55],
}

def search_scenes(bbox, start, end, max_cloud=30, limit=50, sort_by="eo:cloud_cover",
                  min_cloud=None, descending=False):
    """Ask the archive which Sentinel-2 pictures exist over a box and a date range.

    Returns a list of STAC 'items'. Each item is a dictionary of metadata plus
    links to the actual image files. Nothing is downloaded yet.

    Set `min_cloud` when you deliberately want a cloudy scene, which is useful
    for testing that your cloud masking actually works.
    """
    cloud_filter = {"lt": max_cloud}
    if min_cloud is not None:
        cloud_filter["gt"] = min_cloud
    query = {
        "collections": ["sentinel-2-l2a"],
        "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": cloud_filter},
        "limit": limit,
        "sortby": [{"field": f"properties.{sort_by}",
                    "direction": "desc" if descending else "asc"}],
    }
    r = _stac_post(STAC_URL, query, timeout=60)
    return r.json()["features"]

def search_all(bbox, start, end, max_cloud=100, page_size=100, max_pages=20):
    """Every matching scene, not just the first page.

    The archive hands back at most 200 results per request and does not warn you
    that it stopped. This follows the 'next' link until the results run out.
    """
    body = {
        "collections": ["sentinel-2-l2a"], "bbox": bbox,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "query": {"eo:cloud_cover": {"lt": max_cloud}},
        "limit": page_size,
        "sortby": [{"field": "properties.datetime", "direction": "asc"}],
    }
    items, url, pages, matched = [], STAC_URL, 0, None
    while url and pages < max_pages:
        r = _stac_post(url, body, timeout=90)
        j = r.json()
        items += j.get("features", [])
        matched = j.get("context", {}).get("matched", matched)
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        pages += 1
        if not nxt:
            break
        url = nxt[0]["href"]
        body = nxt[0].get("body", body)
    if matched and len(items) < matched:
        print(f"Warning: got {len(items)} of {matched}. Raise max_pages.")
    return items

def make_grid(bbox, metres=20):
    """Define a fixed grid of pixels over a box, in plain latitude and longitude.

    Every image we read gets warped onto this same grid. That is what lets us
    subtract a November picture from an October one pixel by pixel, even when
    the two came from different satellite tiles in different map projections.
    """
    lon0, lat0, lon1, lat1 = bbox
    shrink = math.cos(math.radians((lat0 + lat1) / 2))
    width  = int(round((lon1 - lon0) * 111320 * shrink / metres))
    height = int(round((lat1 - lat0) * 110540 / metres))
    transform = transform_from_bounds(lon0, lat0, lon1, lat1, width, height)
    return {"width": width, "height": height, "transform": transform,
            "metres": metres, "bbox": bbox,
            "pixel_hectares": (metres * metres) / 10000.0}

def read_band(item, band, grid, resampling=Resampling.bilinear):
    """Read one colour band of one scene onto our grid. Returns raw integers."""
    with rasterio.open(item["assets"][band]["href"]) as src:
        with WarpedVRT(src, crs="EPSG:4326", transform=grid["transform"],
                       width=grid["width"], height=grid["height"],
                       resampling=resampling) as vrt:
            return vrt.read(1)

def read_reflectance(item, band, grid):
    """Read a band and convert to reflectance (0 to 1). Divide by 10000."""
    return read_band(item, band, grid).astype("float32") / 10000.0

# Scene Classification Layer codes that mean 'this pixel is usable'.
# 4 vegetation, 5 bare soil, 6 water, 7 low-probability cloud, 11 snow/ice.
CLEAR_CODES = [4, 5, 6, 7, 11]

def clear_mask(item, grid):
    """True where the pixel is usable, False where it is cloud, shadow or edge."""
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return np.isin(scl, CLEAR_CODES)

def check_coverage(item, grid):
    """How much of OUR area this scene actually covers, and how much is clear.

    The cloud percentage in the metadata describes the whole 110 km tile. It
    says nothing about your study area. Always check your own box.
    """
    scl = read_band(item, "scl", grid, Resampling.nearest)
    return {"covered": float((scl > 0).mean()),
            "clear": float(np.isin(scl, CLEAR_CODES).mean())}

def best_scene(items, grid, min_covered=0.95, min_clear=0.60, check_n=8):
    """Walk down the candidate list and return the first scene that is genuinely
    good over our box, not just good on paper."""
    for item in items[:check_n]:
        try:
            c = check_coverage(item, grid)
        except Exception:
            continue
        if c["covered"] >= min_covered and c["clear"] >= min_clear:
            item["_coverage"] = c
            return item
    return None

def composite(items, grid, bands, max_scenes=12, min_clear=0.10, verbose=True):
    """Stack several cloud-masked scenes and take the middle value per pixel.

    One picture of Jamaica almost always has cloud somewhere. Stack ten and take
    the median and the clouds disappear, because cloud is bright and rare while
    the ground underneath is consistent.
    """
    stacks = {b: [] for b in bands}
    used = []
    for item in items:
        if len(used) >= max_scenes:
            break
        try:
            clear = clear_mask(item, grid)
            if clear.mean() < min_clear:
                continue
            for b in bands:
                a = read_reflectance(item, b, grid)
                a[~clear] = np.nan
                a[a <= 0] = np.nan
                stacks[b].append(a)
            used.append(item["properties"]["datetime"][:10])
        except Exception:
            continue
    if not used:
        raise RuntimeError("No usable scenes found. Widen the dates or raise max_cloud.")
    if verbose:
        print(f"Composite built from {len(used)} scenes: {', '.join(sorted(used))}")
    out = {b: np.nanmedian(np.stack(v), axis=0) for b, v in stacks.items()}
    out["_dates"] = sorted(used)
    return out

def normalized_difference(a, b):
    """(a - b) / (a + b). The workhorse formula behind every index in this course."""
    return (a - b) / (a + b + 1e-10)

def stretch(rgb, low=2, high=98):
    """Rescale each colour channel so the picture is bright enough to look at."""
    out = np.zeros_like(rgb, dtype="float32")
    for i in range(rgb.shape[2]):
        band = rgb[:, :, i]
        p1, p2 = np.nanpercentile(band, [low, high])
        out[:, :, i] = np.clip((band - p1) / (p2 - p1 + 1e-9), 0, 1)
    return np.nan_to_num(out)

def show(image, title="", cmap=None, vmin=None, vmax=None, bar=False, size=(9, 8)):
    """Draw an array on screen with sensible defaults."""
    fig, ax = plt.subplots(figsize=size)
    im = ax.imshow(image, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if bar:
        fig.colorbar(im, ax=ax, shrink=0.75)
    plt.tight_layout(); plt.show()

def area_hectares(mask, grid):
    """Convert a True/False mask into hectares on the ground."""
    return float(np.nansum(mask)) * grid["pixel_hectares"]

def label_frame(image_uint8, text):
    """Stamp a label bar onto one animation frame, so every frame says what it is."""
    img = Image.fromarray(image_uint8)
    draw = ImageDraw.Draw(img)
    bar = min(14 + 8 * len(text), img.width)
    draw.rectangle([0, 0, bar, 24], fill=(0, 0, 0))
    draw.text((7, 6), text, fill=(255, 255, 255))
    return np.array(img)

def save_gif(frames, path, ms=900):
    """Write labelled frames out as an animated GIF that loops forever."""
    import imageio.v2 as imageio
    imageio.mimsave(path, frames, duration=ms, loop=0)
    print(f"Saved {path}  ({os.path.getsize(path) / 1e6:.1f} MB, {len(frames)} frames)")

def show_gif(path):
    """Play a GIF inside the notebook."""
    try:
        from IPython.display import Image as _Gif, display
        display(_Gif(filename=path))
    except Exception:
        print("Open the file from the folder panel on the left to watch it.")

print("Toolkit loaded. Study areas available:", ", ".join(PLACES))

In [ ]:
# 🚚 JUST RUN. Five years of measured sunlight at three candidate sites.

def nasa_power_monthly(lat, lon, start, end, parameters, community="AG"):
    # monthly climate records for one point; free NASA POWER service, no account
    # community "RE" (renewable energy) reports sunlight in kWh; "AG" uses megajoules
    url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
    r = requests.get(url, params={
        "parameters": parameters, "community": community,
        "latitude": lat, "longitude": lon,
        "start": start, "end": end, "format": "JSON"}, timeout=120)
    r.raise_for_status()
    block = r.json()["properties"]["parameter"]
    first = parameters.split(",")[0]
    rows = [{"year": int(k[:4]), "month": k[4:], **{p: block[p][k] for p in block}}
            for k in block[first]]
    df = pd.DataFrame(rows)
    return df[(df[list(block)] > -900).all(axis=1)]

sites = {
    "Negril, Westmoreland":   (18.27, -78.35),
    "Santa Cruz, St Elizabeth": (18.05, -77.69),
    "Port Antonio, Portland": (18.18, -76.45),
}

sun = {}
for name, (lat, lon) in sites.items():
    df = nasa_power_monthly(lat, lon, 2020, 2024, "ALLSKY_SFC_SW_DWN", community="RE")
    monthly = df[df.month != "13"].copy()          # month 13 is the annual figure
    monthly["m"] = monthly.month.astype(int)
    sun[name] = monthly.groupby("m")["ALLSKY_SFC_SW_DWN"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
for name, s in sun.items():
    ax.plot(s.index, s.values, "-o", label=f"{name}  (avg {s.mean():.2f})")
ax.set_xlabel("Month"); ax.set_ylabel("Sunlight, kWh per m² per day")
ax.set_title("Measured sunlight at the three sites, 2020 to 2024")
ax.legend(); ax.grid(alpha=0.3); plt.show()

ranked = sorted(sun.items(), key=lambda kv: kv[1].mean(), reverse=True)
for i, (name, s) in enumerate(ranked, 1):
    print(f"{i}. {name}: {s.mean():.2f} kWh/m²/day")
winner_name, winner_sun = ranked[0][0], ranked[0][1].mean()
print(f"\nMEASURED: {winner_name} gets the most sun, {winner_sun:.2f} kWh/m²/day")

---

## Step 2. Your call: the numbers your team must find and defend ✏️

Sunlight is measured. Turning it into powered homes needs four numbers the
satellite cannot know. Split the research.

| Number | Where to look |
|---|---|
| How big is the farm? | Your choice. One football pitch is about 0.7 hectares; real Jamaican solar farms run 10 to 90 hectares |
| What share of the land is actually panels? | Search for solar farm land use: roads, spacing and equipment take roughly half |
| How efficient is a solar panel? | Search for typical solar panel efficiency; also search for the performance ratio of a solar farm |
| What does a Jamaican home use per month? | Search for average JPS (Jamaica Public Service) residential usage in kWh per month |

In [ ]:
# ✏️ YOUR CALL. Change every number, give every number its reason.

team_name = "____"

farm_hectares       = 10       # size of the farm in hectares
panel_share         = 0.50     # share of the land covered by actual panels (0 to 1)
panel_efficiency    = 0.20     # share of sunlight a panel turns into electricity
performance_ratio   = 0.75     # real farms lose some output to heat, dust, wiring
home_kwh_per_month  = 180      # what one Jamaican home uses in a month

reasons = {
    "farm_size":   "____",     # why this size
    "efficiency":  "____",     # where the efficiency and performance ratio came from
    "home_usage":  "____",     # where the household number came from
}

for k, v in reasons.items():
    if "____" in v:
        print(f"⚠️  Fill in your reason for {k}. A number without a reason scores nothing.")

---

## Step 3. Your formula and your answer 🚚 JUST RUN

**daily energy = sunlight × panel area × efficiency × performance ratio**, then
**homes = monthly energy ÷ what one home uses in a month**.

In [ ]:
# 🚚 JUST RUN. From sunshine to homes.
panel_area_m2 = farm_hectares * 10_000 * panel_share
daily_kwh     = winner_sun * panel_area_m2 * panel_efficiency * performance_ratio
monthly_kwh   = daily_kwh * 30
homes         = monthly_kwh / home_kwh_per_month

print(f"Team {team_name} recommends: {winner_name}")
print(f"  Measured sunlight there:  {winner_sun:.2f} kWh/m²/day")
print(f"  Panel area:               {panel_area_m2:,.0f} m² on {farm_hectares} hectares")
print(f"  Electricity produced:     {monthly_kwh:,.0f} kWh per month")
print(f"  Homes that could power:   about {homes:,.0f}")

---

## Step 4. Your 60-second pitch 🚚 JUST RUN

Run the cell, then rehearse it once. The question judges always ask here:
*why not the cloudiest site, and what would change your answer?* Portland's
rain is the clue. Decide your answer before you stand up.

In [ ]:
# 🚚 JUST RUN. Your pitch, written from your own numbers.
runner_up = ranked[1][0]
gap = (ranked[0][1].mean() - ranked[1][1].mean()) / ranked[1][1].mean() * 100
print(f"""
We are team {team_name}.

We compared five years of measured NASA sunlight at three sites.
{winner_name} wins with {winner_sun:.2f} kWh per square metre per day,
about {gap:.0f}% more than {runner_up}.

We sized the farm at {farm_hectares} hectares because {reasons['farm_size']}.
Panel numbers came from {reasons['efficiency']}.
Household usage came from {reasons['home_usage']}.

Our answer: a {farm_hectares}-hectare farm at {winner_name} could power about
{homes:,.0f} homes. The number moves most with panel efficiency and farm size,
so treat it as a scale, not a promise.
""")

---

*Prepared by Adrian Dunkley, Climate Studies Group Mona, Faculty of Science and
Technology, University of the West Indies.*

*Satellite Data Analysis for Jamaica. Built with free, open data: Sentinel-2
from the European Space Agency (ESA), hosted by Amazon; NASA POWER climate
records. No accounts, no fees, no permission needed.*